In [1]:
# Install dependencies
!apt-get update -qq && apt-get install -y -qq zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(10)  # wait for server to fully start
print("Ollama server started")

# Verify it is running
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama is reachable:", r.status_code)
except Exception as e:
    print("ERROR:", e)

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> C

In [2]:
# Pull Mistral-7B (same as your Windows setup)
!ollama pull mistral

# Pull Llama 3-8B (same as your Windows setup)
!ollama pull llama3

# Verify both are available
import requests
r = requests.get("http://localhost:11434/api/tags")
models = [m['name'] for m in r.json().get('models', [])]
print("Available models:", models)



Available models: ['llama3:latest', 'mistral:latest']


In [ ]:
from google.colab import files
import shutil, os

# Upload test.csv from your Windows machine
uploaded = files.upload()   # select data/test.csv when prompted

os.makedirs('/content/smishing_project3/data',    exist_ok=True)
os.makedirs('/content/smishing_project3/outputs', exist_ok=True)
os.makedirs('/content/smishing_project3/src',     exist_ok=True)

# Move uploaded file to correct location
shutil.copy('shap_sample.csv', '/content/smishing_project3/data/shap_sample.csv')
print("shap_sample.csv copied to data folder")

Saving shap_sample.csv to shap_sample (4).csv
shap_sample.csv copied to data folder


In [ ]:
# Upload your llm_agent.py
uploaded = files.upload()   # select src/llm_agent.py when prompted
shutil.copy('llm_agent.py', '/content/smishing_project/src/llm_agent.py')
print("llm_agent.py copied to src folder")

Saving llm_agent.py to llm_agent.py
llm_agent.py copied to src folder


In [ ]:
!pip install scikit-learn pandas numpy requests -q

In [3]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os, pandas as pd
os.chdir("/content/drive/MyDrive/smishing_project3")

for f in ["data/train.csv", "data/val.csv", "data/test.csv"]:
    print(f"\n=== {f} ===")
    if not os.path.exists(f):
        print("  MISSING")
        continue
    df = pd.read_csv(f)
    print(f"  shape    : {df.shape}")
    print(f"  columns  : {df.columns.tolist()}")
    print(f"  balance  : {df['label_binary'].value_counts().to_dict() if 'label_binary' in df.columns else 'no label_binary column'}")
    print(f"  first row: {df.iloc[0].to_dict() if len(df) else 'empty'}")


=== data/train.csv ===
  shape    : (4081, 6)
  columns  : ['text', 'label_binary', 'label_orig', 'has_url', 'has_phone', 'has_email']
  balance  : {0: 3383, 1: 698}
  first row: {'text': 'Someonone you know is trying to contact you via our dating service! To find out who it could be call from your mobile or landline 09064015307 BOX334SK38ch', 'label_binary': 1, 'label_orig': 'smishing', 'has_url': 0, 'has_phone': 1, 'has_email': 0}

=== data/val.csv ===
  shape    : (875, 6)
  columns  : ['text', 'label_binary', 'label_orig', 'has_url', 'has_phone', 'has_email']
  balance  : {0: 725, 1: 150}
  first row: {'text': 'Hi :)finally i completed the course:)', 'label_binary': 0, 'label_orig': 'ham', 'has_url': 0, 'has_phone': 0, 'has_email': 0}

=== data/test.csv ===
  shape    : (875, 6)
  columns  : ['text', 'label_binary', 'label_orig', 'has_url', 'has_phone', 'has_email']
  balance  : {0: 726, 1: 149}
  first row: {'text': 'Ok i msg u b4 i leave my house.', 'label_binary': 0, 'label_ori

In [ ]:
%%javascript
function KeepAlive() {
    document.querySelector("#connect").click();
    console.log("Keeping Colab alive...");
}
setInterval(KeepAlive, 60000);  // clicks every 60 seconds

<IPython.core.display.Javascript object>

In [ ]:
import os
PROJECT = "/content/drive/MyDrive/smishing_project3"
os.chdir(PROJECT)
print("cwd:", os.getcwd())
!ls

cwd: /content/drive/MyDrive/smishing_project3
data  outputs  src


In [ ]:
from pathlib import Path
for p in Path("/").rglob("shap_sample.csv"):
    print(p)

/content/drive/MyDrive/smishing_project3/data/shap_sample.csv
/content/drive/MyDrive/smishing_project/outputs/shap_sample.csv


In [1]:
!pip install -q unsloth trl bitsandbytes

In [ ]:
!python src/llm_agent.py --shap-aligned --configs config2b

LLM SMISHING DETECTION  (input=_100  configs=['config2b'])
  [Ollama] Checking if Ollama is running...
  [Ollama] Ollama is running and responsive.
Available models: ['llama3:latest', 'mistral:latest']

Loading test set from outputs/shap_sample.csv ...
Test messages: 100 (ham=50, malicious=50)
NVIDIA L4, 23034 MiB, 0 MiB, 0 %

MODEL 1/2: mistral
  [Warmup] Loading mistral ...
  [Model] mistral is available.
  [Warmup] Attempt 1/3...
  [Warmup] mistral responded in 2.9s.
NVIDIA L4, 23034 MiB, 4496 MiB, 83 %
NAME              ID              SIZE      PROCESSOR    CONTEXT    UNTIL               
mistral:latest    6577803aa9a0    4.5 GB    100% GPU     512        29 minutes from now    

Config 1/1: config2b
  Description: Decision-first classifier (no CoT)
  Processed: 0/100
  [10/100] Acc: 0.600 | 0.64 msg/s | ETA: 2.3min
  [20/100] Acc: 0.700 | 0.77 msg/s | ETA: 1.7min
  [30/100] Acc: 0.633 | 0.82 msg/s | ETA: 1.4min
  [40/100] Acc: 0.675 | 0.86 msg/s | ETA: 1.2min
  [50/100] Acc: 0.68

In [2]:
import unsloth
print("Unsloth imported OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth imported OK


In [3]:
import trl
import bitsandbytes
import torch
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

trl: 0.24.0
bitsandbytes: 0.50.2
CUDA: True
Device: NVIDIA L4


In [4]:
!cd /content/drive/MyDrive/smishing_project3 && python src/finetune_llm_classifier.py

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
LoRA FINE-TUNING: MISTRAL-7B AND LLAMA-3-8B AS CLASSIFIERS
Train: 4081  |  Val: 875  |  Test: 875
Train balance: {0: 3383, 1: 698}
Test balance:  {0: 726, 1: 149}

TRAINING: mistral  (unsloth/mistral-7b-v0.3-bnb-4bit)
Loading base model ...
==((====))==  Unsloth 2026.9.11: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128.

In [5]:
!pkill -f ollama || true
import time
time.sleep(5)
import unsloth

^C


In [6]:
!cd /content/drive/MyDrive/smishing_project3 && python src/finetune_llm_classifier.py

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
LoRA FINE-TUNING: MISTRAL-7B AND LLAMA-3-8B AS CLASSIFIERS
Train: 4081  |  Val: 875  |  Test: 875
Train balance: {0: 3383, 1: 698}
Test balance:  {0: 726, 1: 149}

TRAINING: llama3  (unsloth/llama-3-8b-Instruct-bnb-4bit)
Loading base model ...
==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128